In [1]:
import numpy as np
from rdkit.Chem import rdFMCS
from rdkit import Chem
from rdkit.Chem import rdDetermineBonds
from rdkit.Chem import AllChem
from rdkit.Geometry import Point3D
import math
from pathlib import Path
import tqdm
import os
from rdkit.Chem import rdmolops



In [2]:
reflections = [
    np.diag([sx, sy, sz])
    for sx in (-1, 1)
    for sy in (-1, 1)
    for sz in (-1, 1)
    if sx * sy * sz == -1
]


def get_coords(mol, conf_id=0):
    conf = mol.GetConformer(conf_id)
    return np.array([
        [conf.GetAtomPosition(i).x,
         conf.GetAtomPosition(i).y,
         conf.GetAtomPosition(i).z]
        for i in range(mol.GetNumAtoms())
    ])

def reflect_coords(coords, R):
    centroid = coords.mean(axis=0)
    return (coords - centroid) @ R.T + centroid

def set_coords(mol, coords, conf_id=0):
    conf = mol.GetConformer(conf_id)
    for i, (x, y, z) in enumerate(coords):
        conf.SetAtomPosition(i, Point3D(float(x), float(y), float(z)))

def xyz_to_rdkit_mol(xyz_file, total_charge):

    with open(xyz_file) as f:
        lines = f.readlines()[2:]  # skip atom count + comment
    atoms, coords = [], []
    for line in lines:
        parts = line.split()
        atoms.append(parts[0])
        coords.append([float(x) for x in parts[1:4]])
    coords = np.array(coords)
    
    # --- Create empty molecule with atoms ---
    mol = Chem.RWMol()
    z = [Chem.GetPeriodicTable().GetAtomicNumber(a) for a in atoms]
    for Zi in z:
        mol.AddAtom(Chem.Atom(Zi))
    
    # --- Add coordinates ---
    conf = Chem.Conformer(len(coords))
    for i, pos in enumerate(coords):
        conf.SetAtomPosition(i, pos)
    mol.AddConformer(conf)
    
    # --- Determine connectivity using RDKit's bond perception ---
    Chem.rdDetermineBonds.DetermineConnectivity(mol, charge=total_charge)
    
    # --- Sanitize molecule ---
    Chem.SanitizeMol(mol)
    
    return mol

In [5]:
from scipy.optimize import linear_sum_assignment

def get_coords(mol):
    conf = mol.GetConformer()
    return np.array([
        conf.GetAtomPosition(i)
        for i in range(mol.GetNumAtoms())
    ])

In [36]:
meci_ref = xyz_to_rdkit_mol('/Users/connerbaucom/Desktop/Pieri/CTG/dim_red_comp/butadiene/MECI/0000_2.xyz', total_charge=0)
unaligned_folder = Path('/Users/connerbaucom/Desktop/Pieri/CTG/dim_red_comp/butadiene/Spawn')



In [38]:
geometries = {}

for x in tqdm.tqdm(list(unaligned_folder.glob('*'))):
    mol2 = xyz_to_rdkit_mol(x, total_charge=0)
    mols = [meci_ref,mol2]
    res=rdFMCS.FindMCS(mols)
    res_smarts= res.smartsString

    mcs = Chem.MolFromSmarts(res_smarts)

    matches1 = meci_ref.GetSubstructMatches(mcs, uniquify=False)
    matches2 = mol2.GetSubstructMatches(mcs, uniquify=False)

    best_rmsd = float("inf")
    best_mol = None

    orig_coords = get_coords(mol2)

    for R in reflections:
        mol2_copy = Chem.Mol(mol2)
        coords = get_coords(mol2_copy)

        reflected = reflect_coords(coords, R)
        set_coords(mol2_copy, reflected)

        for m1 in matches1:
            for m2 in matches2:
                atom_map = list(zip(m2, m1))

                test_mol = Chem.Mol(mol2_copy)
                rmsd = AllChem.AlignMol(
                    test_mol,
                    meci_ref,
                    atomMap=atom_map
                )

                if rmsd < best_rmsd:
                    best_rmsd = rmsd
                    best_mol = test_mol
                    best_atom_map = atom_map





    coords1 = get_coords(meci_ref)
    coords2 = get_coords(best_mol)

    n = meci_ref.GetNumAtoms()
    assert n == best_mol.GetNumAtoms()

    cost = np.zeros((n, n))

    for i in range(n):
        a1 = meci_ref.GetAtomWithIdx(i)
        for j in range(n):
            a2 = best_mol.GetAtomWithIdx(j)

         #Forbid chemically invalid matches
            if a1.GetAtomicNum() != a2.GetAtomicNum():
                cost[i, j] = 1e6
            else:
                cost[i, j] = np.linalg.norm(coords1[i] - coords2[j])


    row_ind, col_ind = linear_sum_assignment(cost)

    new_order = col_ind[np.argsort(row_ind)]
    mol2_reordered = Chem.RenumberAtoms(best_mol, new_order.tolist())
                    
    geometries[x.stem] = mol2_reordered

100%|██████████| 990/990 [00:33<00:00, 29.48it/s]


In [39]:
output = Path('/Users/connerbaucom/Desktop/Pieri/CTG/dim_red_comp/PhotoStats/data/aligned_geometries/mcs/butadiene')
output.mkdir(exist_ok=True, parents=True)
for file, this_mol in geometries.items():
    output_geom = output / (file + ".xyz")
    Chem.MolToXYZFile(this_mol, output_geom)

In [30]:
def combine_xyz_files(input_dir, output_file):


    xyz_files = sorted(
        f for f in os.listdir(input_dir)
        if f.lower().endswith(".xyz")
    )

    if not xyz_files:
        raise ValueError("No .xyz files found in the directory.")

    with open(output_file, "w") as outfile:
        for fname in xyz_files:
            full_path = os.path.join(input_dir, fname)

            with open(full_path) as infile:
                lines = infile.readlines()

            # XYZ files: first line is atom count
            i = 0
            while i < len(lines):
                natoms = int(lines[i].strip())
                frame_end = i + 2 + natoms

                # Write this frame into the combined file
                frame = lines[i:frame_end]
                outfile.writelines(frame)

                i = frame_end

    print(f"Combined {len(xyz_files)} files into {output_file}")

In [40]:
combine_xyz_files(output, "/Users/connerbaucom/Desktop/Pieri/CTG/dim_red_comp/PhotoStats/data/aligned_geometries/mcs/butadiene_combined_spawns.xyz")


Combined 990 files into /Users/connerbaucom/Desktop/Pieri/CTG/dim_red_comp/PhotoStats/data/aligned_geometries/mcs/butadiene_combined_spawns.xyz
